In [11]:
%%capture
%pip install -qU langchain
%pip install -qU langchain-openai
%pip install -qU langchain-community
%pip install python-dotenv
%pip install -qU faiss-cpu
%pip install -qU networkx

In [18]:
!pip show langchain | grep -E 'Name:|Version:'
!pip show langchain-openai | grep -E 'Name:|Version:'
!pip show langchain-community | grep -E 'Name:|Version:'
!pip show python-dotenv | grep -E 'Name:|Version:'
!pip show faiss-cpu | grep -E 'Name:|Version:'
!pip show networkx | grep -E 'Name:|Version:'


Name: langchain
Version: 0.3.27
Name: langchain-openai
Version: 0.3.34
Name: langchain-community
Version: 0.3.30
Name: python-dotenv
Version: 1.1.1
Name: faiss-cpu
Version: 1.12.0
Name: networkx
Version: 3.5


In [12]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.memory import (
    ConversationBufferMemory,
    ConversationSummaryMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryBufferMemory,
    ConversationKGMemory,
    VectorStoreRetrieverMemory
)
from langchain.chains import ConversationChain
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import faiss

In [13]:
load_dotenv()

True

### Conversation Buffer Memory

In [5]:
# Initialize memory
memory = ConversationBufferMemory()

# Create conversation chain
llm = ChatOpenAI(temperature=0.7)
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

# Have a conversation
response1 = conversation.predict(input="Hi, my name is John")
print(f"Response 1: {response1}\n")

response2 = conversation.predict(input="What's my name?")
print(f"Response 2: {response2}\n")

# Check memory contents
print(f"Memory Buffer: {memory.buffer}\n")

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_21272/2299086976.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_21272/2299086976.py:6: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is John
AI:

> Finished chain.
Response 1: Hello John! It's nice to meet you. How can I assist you today?



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is John
AI: Hello John! It's nice to meet you. How can I assist you today?
Human: What's my name?
AI:

> Finished chain.
Response 2: Your name is John, as you mentioned earlier.

Memory Buffer: Human: Hi, my 

### Window Buffer Memory

In [ ]:
memory = ConversationBufferWindowMemory(k=2)
   
llm = ChatOpenAI(temperature=0.7)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# Multiple conversations
conversation.predict(input="Hi, I'm learning Python")
conversation.predict(input="I want to learn about lists")
conversation.predict(input="What about dictionaries?")
conversation.predict(input="Can you explain tuples?")

# Check what's in memory (only last 2 exchanges)
print(f"Window Memory (last 2): {memory.buffer}\n")

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_21272/442229864.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=2)


Window Memory (last 2): Human: What about dictionaries?
AI: Dictionaries are another important data structure in Python. They are unordered collections of key-value pairs, where each key is unique and maps to a specific value. Dictionaries are mutable, meaning you can add, remove, or modify key-value pairs. They are created using curly braces {} and key-value pairs are separated by commas. You can access the value associated with a key using square brackets [] or the get() method. Dictionaries are commonly used to store and retrieve data in a structured way. Would you like me to provide examples or further explanations about dictionaries?
Human: Can you explain tuples?
AI: Tuples are similar to lists in Python, but they are immutable, meaning their elements cannot be changed once they are assigned. Tuples are created using parentheses () and elements are separated by commas. Tuples are often used to store related pieces of information together, such as coordinates or RGB color values. 

### Conversation Summary Memory

In [7]:
llm = ChatOpenAI(temperature=0)

memory = ConversationSummaryMemory(llm=llm)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# Have a longer conversation
conversation.predict(input="I'm working on a data science project")
conversation.predict(input="I need to analyze customer behavior data")
conversation.predict(input="The dataset has 1 million rows and 50 features")
conversation.predict(input="I'm considering using random forest or XGBoost")

# Get the summary
print(f"Conversation Summary: {memory.buffer}\n")
   

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_21272/4126921006.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm)


Conversation Summary: The human mentions working on a data science project analyzing customer behavior data with a dataset containing 1 million rows and 50 features. The AI expresses enthusiasm for the project and asks about specific aspects like purchase history, website interactions, and demographic information that could provide valuable insights for businesses. The AI also comments on the large dataset, suggesting there must be a lot of valuable information to work with and inquires about any specific patterns or trends in the data that have been identified so far. The human considers using random forest or XGBoost for the project, and the AI provides insights into the differences between the two algorithms, highlighting their strengths in handling large datasets and the importance of considering trade-offs in model complexity, interpretability, and computational resources.



### Summary buffer memory

In [8]:
llm = ChatOpenAI(temperature=0)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=200  # Summarize when exceeding token limit
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# Multiple exchanges
topics = [
    "I want to build a machine learning model",
    "The model should predict customer churn",
    "I have historical data from the past 2 years",
    "Features include usage patterns, demographics, and support tickets",
    "What algorithm would you recommend?",
    "How should I handle class imbalance?"
]

for topic in topics:
    response = conversation.predict(input=topic)
    print(f"User: {topic}")
    print(f"AI: {response[:100]}...\n")  # Truncate for display

print(f"Memory State: {memory.buffer}\n")
   

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_21272/3557103004.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(


User: I want to build a machine learning model
AI: That's a great idea! Machine learning models are powerful tools for analyzing and making predictions...

User: The model should predict customer churn
AI: Predicting customer churn is a common use case for machine learning models in business. To build a m...

User: I have historical data from the past 2 years
AI: That's great! Historical data is crucial for building an accurate customer churn prediction model. M...

User: Features include usage patterns, demographics, and support tickets
AI: Those are important features that can provide valuable insights into customer behavior and potential...

User: What algorithm would you recommend?
AI: There are several algorithms that can be used for customer churn prediction, such as logistic regres...

User: How should I handle class imbalance?
AI: Handling class imbalance is crucial in building an accurate customer churn prediction model. There a...

Memory State: System: The human expresses in

### Kmowledge Graph Memory

In [14]:
llm = ChatOpenAI(temperature=0)

memory = ConversationKGMemory(llm=llm)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# Conversation with entities
conversation.predict(input="Emma works at Google as a software engineer")
conversation.predict(input="She specializes in machine learning and Python")
conversation.predict(input="Emma graduated from MIT in 2020")
conversation.predict(input="What do you know about Emma?")

# Get knowledge graph
kg = memory.kg.get_triples()
print("Knowledge Graph Triples:")
for triple in kg:
    print(f"  {triple}")
print()

Knowledge Graph Triples:
  ('Emma', 'Google', 'works at')
  ('Emma', 'software engineer', 'is a')
  ('Emma', 'machine learning', 'specializes in')
  ('Emma', 'Python', 'specializes in')
  ('Emma', 'MIT', 'graduated from')
  ('Emma', '2020', 'graduated in')
  ('Emma', 'machine learning and Python', 'specializes in')



### Vector store memory

In [15]:
 # Create embeddings and vector store
embeddings = OpenAIEmbeddings()

# Create an empty vector store
embedding_size = 1536  # OpenAI embeddings size
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(
    embedding_function=embeddings.embed_query,
    index=index,
    docstore=faiss.DocStore(),
    index_to_docstore_id={}
)

# Create retriever memory
retriever = vectorstore.as_retriever(search_kwargs=dict(k=2))
memory = VectorStoreRetrieverMemory(retriever=retriever)

# Add some memories
memory.save_context(
    {"input": "I love programming in Python"},
    {"output": "Python is a great language for beginners and experts alike"}
)
memory.save_context(
    {"input": "Machine learning is fascinating"},
    {"output": "Yes, ML opens up many possibilities in data analysis"}
)
memory.save_context(
    {"input": "I enjoy hiking on weekends"},
    {"output": "Hiking is a great way to stay active and enjoy nature"}
)

# Search relevant memories
relevant_memories = memory.retriever.get_relevant_documents("Tell me about Python")
print("Relevant Memories for 'Python':")
for mem in relevant_memories:
    print(f"  - {mem.page_content}")
print()

/var/folders/wy/mmq3bg9s30s2zc02dq4r60z80000gn/T/ipykernel_21272/2141638674.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


AttributeError: module 'faiss' has no attribute 'DocStore'

### Custom Memory


In [20]:
class CustomMemoryManager:
    """Custom memory management example"""
   
    def __init__(self):
        self.short_term = ConversationBufferWindowMemory(k=3)
        self.long_term = ConversationSummaryMemory(llm=ChatOpenAI(temperature=0))
        self.important_facts = []
   
    def add_interaction(self, user_input, ai_output):
        """Add interaction to both memories"""
        # Add to short-term
        self.short_term.save_context(
            {"input": user_input},
            {"output": ai_output}
        )
       
        # Add to long-term
        self.long_term.save_context(
            {"input": user_input},
            {"output": ai_output}
        )
       
        # Extract important facts (simplified)
        if "important" in user_input.lower() or "remember" in user_input.lower():
            self.important_facts.append({
                "user": user_input,
                "ai": ai_output
            })
   
    def get_context(self):
        """Get combined context"""
        return {
            "recent": self.short_term.buffer,
            "summary": self.long_term.buffer,
            "facts": self.important_facts
        }
 
def custom_memory_example():
    """Custom memory management"""
    print("=== Custom Memory Manager Example ===")
   
    manager = CustomMemoryManager()
   
    # Add interactions
    interactions = [
        ("My name is Alice", "Nice to meet you, Alice!"),
        ("I'm learning LangChain", "LangChain is great for building LLM applications"),
        ("Remember: my favorite color is blue", "I'll remember that your favorite color is blue"),
        ("What are we discussing?", "We're discussing LangChain and I know your favorite color is blue")
    ]
   
    for user, ai in interactions:
        manager.add_interaction(user, ai)
   
    # Get context
    context = manager.get_context()
    print("Memory Context:")
    print(f"Recent: {context['recent'][:200]}...")
    print(f"Summary: {context['summary'][:200]}...")
    print(f"Important Facts: {context['facts']}")
    print()
   
custom_memory_example()

=== Custom Memory Manager Example ===
Memory Context:
Recent: Human: I'm learning LangChain
AI: LangChain is great for building LLM applications
Human: Remember: my favorite color is blue
AI: I'll remember that your favorite color is blue
Human: What are we disc...
Summary: Alice introduces herself to the AI, who responds warmly. Alice mentions she is learning LangChain, and the AI praises LangChain for building LLM applications. The human shares their favorite color is ...
Important Facts: [{'user': 'Remember: my favorite color is blue', 'ai': "I'll remember that your favorite color is blue"}]

